# Quran Search Engine

| Phase | Sections | Description |
|-------|----------|-------------|
| **Phase 1** | 1 – 5 | Setup · Load · Corpus · Preprocessing · Inverted Index |
| **Phase 2** | 6 – 9 | TF-IDF Ranking · AND Retrieval · Rocchio PRF · BERT Expansion · Unified API |

**Full pipeline:**
```
quran.json  (114 surahs, 6 236 ayahs)
  └─▶  Corpus Builder  ──  flatten → 6 236 ayah documents
         └─▶  Preprocessing  ──  diacritics → normalize → tokenize → stopwords → ISRI stem
                └─▶  Inverted Index  ──  term → { df, postings: {doc_id: tf} }
                       └─▶  TF-IDF ranked retrieval  (OR union)
                              └─▶  AND retrieval  (posting-list intersection)
                                     └─▶  Rocchio PRF  (centroid expansion)
                                            └─▶  BERT Semantic Expansion  (multilingual embeddings)
```

---
## Section 1 — Setup & Dependencies

In [1]:
!pip install nltk pyarabic sentence-transformers --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 1.9 MB/s eta 0:00:00 0:00:01


In [2]:
import json
import re
import os
import math
import urllib.request
import numpy as np
from collections import defaultdict
from typing import List, Dict, Tuple, Set

import nltk
from nltk.stem.isri import ISRIStemmer
nltk.download('stopwords', quiet=True)

print('All imports successful')

All imports successful


---
## Section 2 — Load Data

Loads `quran.json` from a local path **or** downloads it from GitHub if not present.

Expected JSON structure:
```json
[
  {
    "id": 1, "name": "الفاتحة", "transliteration": "Al-Fatiha",
    "type": "meccan", "total_verses": 7,
    "verses": [ {"id": 1, "text": "بِسۡمِ ٱللَّهِ..."}, ... ]
  }, ...
]
```

In [3]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Update this URL to point to the raw quran.json in your GitHub repository
GITHUB_RAW_QURAN = (
    'https://raw.githubusercontent.com/YOUR_USERNAME/YOUR_REPO/main/quran.json'
)
QURAN_PATH   = 'quran.json'
OUTPUT_DIR   = 'quran_ir_output'


def _download_if_missing(local_path: str, url: str) -> None:
    if not os.path.exists(local_path):
        print(f'Downloading {os.path.basename(local_path)} from GitHub...')
        urllib.request.urlretrieve(url, local_path)
        print('  Done.')


def load_quran(path: str) -> List[Dict]:
    """Load and sanity-check the Quran JSON file."""
    assert os.path.exists(path), f'File not found: {path}'
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    assert isinstance(data, list), 'Top-level must be a list of surahs'
    assert len(data) == 114,       f'Expected 114 surahs, got {len(data)}'
    return data


def validate_structure(data: List[Dict]) -> None:
    """Validate required keys and non-empty verse text across every ayah."""
    SURAH_KEYS = {'id', 'name', 'verses'}
    VERSE_KEYS = {'id', 'text'}
    errors: List[str] = []
    for surah in data:
        if not SURAH_KEYS.issubset(surah):
            errors.append(f"Surah {surah.get('id')} missing keys: {SURAH_KEYS - surah.keys()}")
            continue
        for verse in surah['verses']:
            if not VERSE_KEYS.issubset(verse):
                errors.append(f"[{surah['id']}:{verse.get('id')}] missing keys")
            elif not isinstance(verse.get('text', ''), str) or not verse['text'].strip():
                errors.append(f"[{surah['id']}:{verse.get('id')}] empty/invalid text")
    if errors:
        for e in errors[:5]: print(e)
        raise ValueError(f'{len(errors)} validation error(s).')
    print('Structure validation passed — no errors')


# ── Load ──────────────────────────────────────────────────────────────────────
_download_if_missing(QURAN_PATH, GITHUB_RAW_QURAN)
raw_data = load_quran(QURAN_PATH)
validate_structure(raw_data)

print()
print('Dataset Overview')
print(f'  Surahs        : {len(raw_data)}')
print(f'  Total ayahs   : {sum(s["total_verses"] for s in raw_data):,}')
print(f'  Meccan surahs : {sum(1 for s in raw_data if s.get("type","").lower() in ("meccan","makkah"))}')
print(f'  Medinan surahs: {sum(1 for s in raw_data if s.get("type","").lower() in ("medinan","madinah"))}')

HTTPError: HTTP Error 404: Not Found

In [ ]:
# Print first surah as a structural sample
sample = raw_data[0]
print(f"Surah  : {sample['id']} — {sample['name']} ({sample.get('transliteration','')}")
print(f"Type   : {sample.get('type','')}   |   Verses: {sample['total_verses']}")
print()
for v in sample['verses']:
    print(f"  [{sample['id']}:{v['id']}]  {v['text']}")

---
## Section 3 — Build Corpus

Each ayah becomes one document:
```json
{ "doc_id": 1, "surah_id": 1, "surah_name": "الفاتحة", "ayah_id": 1, "text": "بِسۡمِ ٱللَّهِ..." }
```

In [ ]:
def build_corpus(data: List[Dict]) -> List[Dict]:
    """Flatten all surahs into a list of ayah documents with sequential doc_id."""
    corpus: List[Dict] = []
    doc_id = 1
    for surah in data:
        for verse in surah['verses']:
            text = verse['text'].strip()
            if not text:
                continue
            corpus.append({
                'doc_id'    : doc_id,
                'surah_id'  : surah['id'],
                'surah_name': surah['name'],
                'ayah_id'   : verse['id'],
                'text'      : text,
            })
            doc_id += 1
    return corpus


corpus = build_corpus(raw_data)

print(f'Corpus built: {len(corpus):,} documents')
print()
print('Sample documents:')
for doc in corpus[:4]:
    print(f"  doc_id={doc['doc_id']:4d}  [{doc['surah_id']}:{doc['ayah_id']}]  {doc['text']}")

---
## Section 4 — Arabic Preprocessing

| Step | Function | What it does |
|------|----------|--------------|
| 1 | `remove_diacritics` | Strip tashkeel, harakat, tatweel (U+064B–U+065F, U+0640, U+0670, U+06D6–U+06ED) |
| 2 | `normalize_arabic` | أ/إ/آ/ٱ→ا · ة→ه · ى/ئ→ي · ؤ→و |
| 3 | `tokenize` | Whitespace split, remove non-Arabic characters |
| 4 | `remove_stopwords` | Drop Arabic function words (691 terms incl. NLTK list) |
| 5 | `stem_words` | ISRI light Arabic stemmer (prefix/suffix stripping) |
| — | `preprocess` | Full pipeline in one call |

In [ ]:
# ── Compiled regex patterns ───────────────────────────────────────────────────
#  Harakat/Tashkeel : U+064B–U+065F
#  Superscript alef : U+0670
#  Extended diacr.  : U+06D6–U+06DC, U+06DF–U+06E4, U+06E7, U+06E8, U+06EA–U+06ED
#  Tatweel (kashida): U+0640
#  Arabic core      : U+0621–U+064A
_DIACRITICS_RE = re.compile(
    r'[\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E4\u06E7\u06E8\u06EA-\u06ED\u0640]'
)
_NON_ARABIC_RE = re.compile(r'[^\u0621-\u064A\s]')


def remove_diacritics(text: str) -> str:
    """Remove all tashkeel, harakat, and tatweel."""
    return _DIACRITICS_RE.sub('', text)


def normalize_arabic(text: str) -> str:
    """Normalise Arabic letter variants to canonical base forms."""
    text = re.sub(r'[أإآٱ]', 'ا', text)
    text = re.sub(r'ة',       'ه', text)
    text = re.sub(r'[ىئ]',   'ي', text)
    return re.sub(r'ؤ',       'و', text)


def tokenize(text: str) -> List[str]:
    """Replace non-Arabic characters with spaces then split on whitespace."""
    return [tok for tok in _NON_ARABIC_RE.sub(' ', text).split() if tok]


# ── Arabic Stopwords ──────────────────────────────────────────────────────────
_HARDCODED_SW: Set[str] = {
    # Prepositions & conjunctions
    'من','إلى','عن','على','في','مع','عند','بين','قبل','بعد','فوق','تحت',
    'حول','خلال','إلا','غير','دون','ضد','نحو','حتى','منذ','رغم',
    # Pronouns
    'هو','هي','هم','هن','نحن','أنت','أنتم','أنتن','أنا','هما','أنتما',
    'هذا','هذه','ذلك','تلك','هؤلاء','أولئك','هنا','هناك','ثمة',
    # Relative pronouns
    'الذي','التي','الذين','اللاتي','اللتان','اللذان','اللواتي',
    # Conjunctions & particles
    'و','أو','أم','لكن','بل','ثم','فـ','واو','إذ','إذا','لو','لولا','لما',
    'أن','إن','كأن','لأن','لكي','كي','حين','عندما','بينما','منذ',
    'قد','لم','لن','لا','ما','ليس','ليست','ألا','أما','إما',
    'ها','يا','أيها','أيتها','سوف','سـ',
    # Auxiliary verbs
    'كان','كانت','كانوا','يكون','تكون','يكونوا','تكونوا',
    'صار','صارت','أصبح','أصبحت','بات','باتت','ظل','ظلت','بقي','بقيت',
    # Attached pronoun forms (post-normalization)
    'له','لها','لهم','لنا','لكم','لكن',
    'به','بها','بهم','بنا','بكم',
    'منه','منها','منهم','منا','منكم',
    'عليه','عليها','عليهم','علينا','عليكم',
    'إليه','إليها','إليهم','إلينا','إليكم',
    'عنه','عنها','عنهم','عنا','عنكم',
    'فيه','فيها','فيهم','فينا','فيكم',
    'معه','معها','معهم','معنا','معكم',
    # High-frequency Quranic particles
    'ذو','ذات','ذوو','ذوات','أي','أيها','كم','أين','كيف','متى','لماذا',
    'كل','كلا','كلما','جميع','بعض','كثير','قليل',
    'أيضا','جدا','نفس','أحد','إحدى',
    # Waw / Fa-prefixed high-frequency forms
    'وما','وهو','وهي','وهم','وكان','وكانت','وقد','ولا','ولم','وإن','وأن',
    'فما','فهو','فهي','فهم','فقد','فلا','فلم','فكان','فكانت','فإن','فأن',
    # إنَّ family
    'إنه','إنها','إنهم','إنك','إننا','إنكم',
    'انه','انها','انهم','انك','اننا','انكم','انما',
}

def _nw(w: str) -> str:
    return normalize_arabic(remove_diacritics(w))

ARABIC_STOPWORDS: Set[str] = {_nw(w) for w in _HARDCODED_SW}
try:
    from nltk.corpus import stopwords as _nltk_sw
    ARABIC_STOPWORDS |= {_nw(w) for w in _nltk_sw.words('arabic')}
    print(f'NLTK stopwords merged  →  total: {len(ARABIC_STOPWORDS)} stopwords')
except Exception:
    print(f'Using hardcoded list only: {len(ARABIC_STOPWORDS)} stopwords')


def remove_stopwords(tokens: List[str]) -> List[str]:
    """Filter Arabic stopwords. Input must already be normalised."""
    return [t for t in tokens if t not in ARABIC_STOPWORDS]


_stemmer = ISRIStemmer()

def stem_words(tokens: List[str]) -> List[str]:
    """Apply ISRI light Arabic stemmer (removes prefixes/suffixes without a lexicon)."""
    return [_stemmer.stem(t) for t in tokens]


def preprocess(text: str, apply_stemming: bool = True) -> Tuple[str, List[str]]:
    """
    Full Arabic preprocessing pipeline.

    Steps: remove_diacritics → normalize_arabic → tokenize
           → remove_stopwords → stem_words (optional)

    Returns:
        normalized_text : str          (after steps 1–2, before tokenization)
        tokens          : List[str]
    """
    text = remove_diacritics(text)
    text = normalize_arabic(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    if apply_stemming:
        tokens = stem_words(tokens)
    return text, tokens


print('Preprocessing functions defined')

In [ ]:
# ── Step-by-step demo on Al-Fatiha verse 1 ────────────────────────────────────
demo_text = corpus[0]['text']
print('Original         :', demo_text)
step1 = remove_diacritics(demo_text)
print('After diacritics :', step1)
step2 = normalize_arabic(step1)
print('After normalize  :', step2)
step3 = tokenize(step2)
print('After tokenize   :', step3)
step4 = remove_stopwords(step3)
print('After stopwords  :', step4)
step5 = stem_words(step4)
print('After stemming   :', step5)

In [ ]:
def process_corpus(corpus: List[Dict], apply_stemming: bool = True) -> List[Dict]:
    """Enrich every document with normalized_text and tokens fields."""
    processed: List[Dict] = []
    for doc in corpus:
        normed, tokens = preprocess(doc['text'], apply_stemming=apply_stemming)
        processed.append({**doc, 'normalized_text': normed, 'tokens': tokens})
    return processed


processed_corpus = process_corpus(corpus, apply_stemming=True)

print(f'Preprocessing complete: {len(processed_corpus):,} documents')
print()
for doc in processed_corpus[:3]:
    print(f"  [{doc['surah_id']}:{doc['ayah_id']}]")
    print(f"    Original        : {doc['text']}")
    print(f"    Normalized text : {doc['normalized_text']}")
    print(f"    Tokens          : {doc['tokens']}")
    print()

In [ ]:
# ── Corpus statistics ─────────────────────────────────────────────────────────
all_tokens   = [tok for doc in processed_corpus for tok in doc['tokens']]
token_freq   = defaultdict(int)
for t in all_tokens: token_freq[t] += 1

empty_docs   = [d for d in processed_corpus if not d['tokens']]
top_10       = sorted(token_freq.items(), key=lambda x: x[1], reverse=True)[:10]

print('Corpus Statistics')
print('─' * 45)
print(f'  Total documents         : {len(processed_corpus):,}')
print(f'  Total tokens (all docs) : {len(all_tokens):,}')
print(f'  Vocabulary size         : {len(token_freq):,} unique terms')
print(f'  Avg tokens per document : {len(all_tokens)/len(processed_corpus):.1f}')
print(f'  Documents with 0 tokens : {len(empty_docs)}')
print()
print(f'  {"Top 10 terms after preprocessing":}')
print(f'  {"Term":<20} Freq')
print('  ' + '─' * 28)
for term, freq in top_10:
    print(f'  {term:<20} {freq:,}')

---
## Section 5 — Build & Save Inverted Index

**Structure per entry:**
```json
"term": {
  "df": 42,
  "postings": { "doc_id": tf, ... }    // sorted by doc_id
}
```

| Saved file | Contents | Used in |
|------------|----------|---------|
| `c_quran.json` | Full preprocessed corpus | Phase 2 |
| `inverted_index.json` | term → { df, postings } | Phase 2 ranking |
| `vocab.json` | term → integer ID | Phase 2 vectorisation |
| `doc_metadata.json` | Lightweight display table | Phase 3 UI |

In [ ]:
def build_inverted_index(processed_corpus: List[Dict]) -> Dict:
    """
    Build inverted index from preprocessed corpus.
    Postings lists are sorted ascending by doc_id for efficient merge.
    """
    index: Dict = defaultdict(lambda: {'df': 0, 'postings': {}})

    for doc in processed_corpus:
        doc_id = str(doc['doc_id'])
        tf_map: Dict[str, int] = defaultdict(int)
        for token in doc['tokens']:
            tf_map[token] += 1
        for term, tf in tf_map.items():
            if doc_id not in index[term]['postings']:
                index[term]['df'] += 1
            index[term]['postings'][doc_id] = tf

    # Sort each postings list by doc_id (numeric)
    return {
        term: {
            'df'      : entry['df'],
            'postings': dict(sorted(entry['postings'].items(), key=lambda x: int(x[0])))
        }
        for term, entry in index.items()
    }


inverted_index = build_inverted_index(processed_corpus)

print(f'Inverted index built  →  {len(inverted_index):,} unique terms')
print()

# Index health stats
df_values = [v['df'] for v in inverted_index.values()]
hapax     = sum(1 for d in df_values if d == 1)
top_df    = sorted(inverted_index.items(), key=lambda x: x[1]['df'], reverse=True)[:10]

print('Inverted Index Statistics')
print('─' * 45)
print(f'  Unique terms         : {len(inverted_index):,}')
print(f'  Hapax legomena (df=1): {hapax:,} ({100*hapax/len(inverted_index):.1f}%)')
print(f'  Max df               : {max(df_values)}')
print(f'  Mean df              : {sum(df_values)/len(df_values):.2f}')
print()
print(f'  {"Top 10 terms by document frequency":}')
print(f'  {"Term":<20} df')
print('  ' + '─' * 28)
for term, entry in top_df:
    print(f'  {term:<20} {entry["df"]}')

In [ ]:
def lookup_term(raw_term: str, top_k: int = 5) -> None:
    """Preprocess a raw term and show its posting list entry."""
    _, stemmed = preprocess(raw_term)
    if not stemmed:
        print(f"'{raw_term}' → removed as stopword/empty")
        return
    term  = stemmed[0]
    entry = inverted_index.get(term)
    if not entry:
        print(f"'{raw_term}' → stemmed '{term}' → NOT FOUND")
        return
    top_p = dict(list(entry['postings'].items())[:top_k])
    print(f"'{raw_term}'  →  stemmed: '{term}'  |  df={entry['df']}  |  postings (first {top_k}): {top_p}")


for word in ['الله', 'رحمة', 'نور', 'كتاب', 'صلاة', 'يوم', 'قلب']:
    lookup_term(word)

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

def _save_json(obj: object, fname: str) -> None:
    path = os.path.join(OUTPUT_DIR, fname)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    print(f'  Saved {fname:<35} {os.path.getsize(path)/1024:8.1f} KB')


print('Saving Phase 1 artifacts...')
_save_json(processed_corpus, 'c_quran.json')
_save_json(inverted_index,    'inverted_index.json')
_save_json(
    {term: idx for idx, term in enumerate(sorted(inverted_index))},
    'vocab.json'
)
_save_json(
    [{'doc_id': d['doc_id'], 'surah_id': d['surah_id'],
      'surah_name': d['surah_name'], 'ayah_id': d['ayah_id'], 'text': d['text']}
     for d in processed_corpus],
    'doc_metadata.json'
)
print(f'\nAll artifacts saved to ./{OUTPUT_DIR}/')

---
## Section 6 — TF-IDF Ranked Retrieval

Uses the inverted index from Phase 1 directly — no external IR library needed.

**Scoring formula:**

$$\text{score}(q,d) = \sum_{t\,\in\,q} \underbrace{\text{tf}(t,d)}_{\text{raw count}} \times \underbrace{\log\!\left(\frac{N+1}{\text{df}(t)+1}\right)}_{\text{smoothed IDF}}$$

Two retrieval modes:
- **OR (union)** — scores every document that contains *any* query term, ranked by TF-IDF sum.
- **AND (intersection)** — restricts candidates to documents containing *all* query terms, then ranks by TF-IDF.

In [ ]:
# ── Lookup structures ─────────────────────────────────────────────────────────
N          = len(processed_corpus)                          # total documents
DOC_LOOKUP = {d['doc_id']: d for d in processed_corpus}    # int → doc dict


def _tfidf(tf: int, df: int) -> float:
    """TF × smoothed IDF for one (term, document) pair."""
    return tf * math.log((N + 1) / (df + 1))


def _score_docs(query_tokens: List[str], candidate_ids: Set[int] = None) -> List[Tuple[int,float]]:
    """
    Accumulate TF-IDF scores for query_tokens.
    If candidate_ids is given, only those documents are scored (AND mode).
    Returns list of (doc_id, score) sorted descending.
    """
    scores: Dict[int, float] = defaultdict(float)
    for term in set(query_tokens):
        entry = inverted_index.get(term)
        if not entry:
            continue
        df = entry['df']
        for did_str, tf in entry['postings'].items():
            did = int(did_str)
            if candidate_ids is None or did in candidate_ids:
                scores[did] += _tfidf(tf, df)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def _to_result_list(ranked: List[Tuple[int,float]], top_k: int) -> List[Dict]:
    """Convert (doc_id, score) pairs into full result dicts."""
    return [
        {**DOC_LOOKUP[did], 'score': round(score, 4)}
        for did, score in ranked[:top_k]
    ]


def search_or(query: str, top_k: int = 10) -> List[Dict]:
    """TF-IDF OR retrieval: union of all postings, ranked by score."""
    _, q_tokens = preprocess(query)
    if not q_tokens:
        print('Query empty after preprocessing.')
        return []
    return _to_result_list(_score_docs(q_tokens), top_k)


def search_and(query: str, top_k: int = 10) -> List[Dict]:
    """
    TF-IDF AND retrieval: intersect posting lists (sorted ascending by size
    for early pruning), then re-rank by TF-IDF.
    Falls back to OR if intersection is empty.
    """
    _, q_tokens = preprocess(query)
    if not q_tokens:
        print('Query empty after preprocessing.')
        return []

    # Build one posting set per distinct query term
    posting_sets = [
        {int(k) for k in inverted_index[t]['postings']}
        for t in set(q_tokens) if t in inverted_index
    ]
    if not posting_sets:
        return []

    # Intersect in ascending-size order
    posting_sets.sort(key=len)
    and_ids = posting_sets[0]
    for ps in posting_sets[1:]:
        and_ids &= ps
        if not and_ids:
            break

    if not and_ids:
        print('No AND match — falling back to OR.')
        return search_or(query, top_k)

    return _to_result_list(_score_docs(q_tokens, and_ids), top_k)


def display_results(results: List[Dict], title: str = 'Results') -> None:
    """Print search results as a readable table."""
    print(f'\n── {title} ({len(results)} hits) ──────────────────────────────')
    print(f"  {'#':>2}  {'Ref':<10}  {'Surah':<16}  {'Score':>7}  Text")
    print('  ' + '─' * 95)
    for i, r in enumerate(results, 1):
        ref  = f"[{r['surah_id']}:{r['ayah_id']}]"
        text = r['text'][:68] + ('…' if len(r['text']) > 68 else '')
        print(f"  {i:>2}  {ref:<10}  {r['surah_name']:<16}  {r['score']:>7.4f}  {text}")


# ── Demo ──────────────────────────────────────────────────────────────────────
demo_q = 'الله الرحمة النور'
display_results(search_or(demo_q,  top_k=10), f'TF-IDF OR  |  "{demo_q}"')
display_results(search_and('الصبر الصلاة', top_k=10), 'TF-IDF AND  |  "الصبر الصلاة"')

---
## Section 7 — Rocchio Pseudo Relevance Feedback (PRF)

**Algorithm:**

$$\vec{q}_{\text{new}} = \alpha\,\vec{q}_{\text{orig}}\;+\;\beta\,\frac{1}{|R|}\sum_{d\in R}\vec{d}$$

In practice: keep original query terms **plus** append the highest-TF-IDF-weighted terms
from the top-$k$ retrieved documents that do **not** already appear in the query.

In [ ]:
def rocchio_expand(
    query   : str,
    k_rel   : int   = 5,
    n_terms : int   = 5,
    beta    : float = 0.75,
) -> Tuple[List[str], List[str]]:
    """
    Expand query using Rocchio PRF.

    Args:
        query   : original Arabic query
        k_rel   : top-k docs to treat as pseudo-relevant
        n_terms : max new terms to add
        beta    : centroid weight

    Returns:
        (expanded_token_list, new_terms_added)
    """
    _, q_tokens = preprocess(query)
    if not q_tokens:
        return [], []

    # Step 1 — retrieve top-k pseudo-relevant documents
    rel_docs = _to_result_list(_score_docs(q_tokens), top_k=k_rel)
    if not rel_docs:
        return q_tokens, []

    # Step 2 — build TF-IDF-weighted centroid over relevant docs
    centroid: Dict[str, float] = defaultdict(float)
    for doc in rel_docs:
        doc_tokens = DOC_LOOKUP[doc['doc_id']]['tokens']   # already stemmed
        tf_map: Dict[str, int] = defaultdict(int)
        for t in doc_tokens: tf_map[t] += 1
        for term, tf in tf_map.items():
            df = inverted_index.get(term, {}).get('df', 1)
            centroid[term] += beta * _tfidf(tf, df) / k_rel

    # Step 3 — pick top new terms not already in original query
    q_set    = set(q_tokens)
    new_terms = [
        t for t, _ in
        sorted(
            ((t, w) for t, w in centroid.items() if t not in q_set),
            key=lambda x: x[1], reverse=True
        )[:n_terms]
    ]
    return q_tokens + new_terms, new_terms


def search_rocchio(query: str, k_rel: int = 5, n_terms: int = 5, top_k: int = 10) -> List[Dict]:
    """Rocchio PRF: expand query → re-rank with TF-IDF."""
    expanded_tokens, new_terms = rocchio_expand(query, k_rel, n_terms)
    _, q_orig = preprocess(query)
    print(f'  Original tokens : {q_orig}')
    print(f'  Added terms     : {new_terms}')
    return _to_result_list(_score_docs(expanded_tokens), top_k)


# ── Demo ──────────────────────────────────────────────────────────────────────
display_results(
    search_rocchio('الصبر الصلاة', top_k=10),
    'Rocchio PRF  |  "الصبر الصلاة"'
)

---
## Section 8 — BERT Semantic Query Expansion

Uses **`paraphrase-multilingual-MiniLM-L12-v2`** (supports Arabic).

**Steps:**
1. Encode preprocessed query → L2-normalised vector.
2. Encode all vocabulary terms once and cache as `VOCAB_VECS`.
3. Cosine similarity = dot product (both normalised).
4. Append top-K similar terms (above threshold) not already in the query.
5. Re-rank expanded term set with TF-IDF.

In [ ]:
from sentence_transformers import SentenceTransformer

SBERT = SentenceTransformer(
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
)

# Pre-encode all vocabulary terms once and cache
VOCAB_TERMS = list(inverted_index.keys())
print(f'Encoding {len(VOCAB_TERMS):,} vocabulary terms — may take ~30 s on CPU …')
VOCAB_VECS  = SBERT.encode(
    VOCAB_TERMS, batch_size=512, show_progress_bar=True, normalize_embeddings=True
)   # shape: (|V|, 384), L2-normalised
print('Vocabulary embeddings cached.')

In [ ]:
def bert_expand(
    query     : str,
    k_terms   : int   = 5,
    threshold : float = 0.55,
) -> Tuple[List[str], List[str]]:
    """
    Expand query using multilingual BERT cosine similarity.

    Args:
        query     : original Arabic query
        k_terms   : max semantically similar terms to add
        threshold : minimum cosine similarity (0–1)

    Returns:
        (expanded_token_list, new_terms_added)
    """
    _, q_tokens = preprocess(query)
    if not q_tokens:
        return [], []

    # Encode query (L2-normalised)
    q_vec = SBERT.encode(' '.join(q_tokens), normalize_embeddings=True)  # (384,)

    # Cosine similarities: dot product with pre-normalised vocab vectors
    sims = VOCAB_VECS @ q_vec   # (|V|,)

    q_set     = set(q_tokens)
    new_terms = []
    for idx in np.argsort(sims)[::-1]:
        if len(new_terms) >= k_terms:
            break
        term = VOCAB_TERMS[idx]
        if term not in q_set and sims[idx] >= threshold:
            new_terms.append(term)

    return q_tokens + new_terms, new_terms


def search_bert(
    query     : str,
    k_terms   : int   = 5,
    threshold : float = 0.55,
    top_k     : int   = 10,
) -> List[Dict]:
    """BERT semantic expansion + TF-IDF retrieval."""
    expanded_tokens, new_terms = bert_expand(query, k_terms, threshold)
    _, q_orig = preprocess(query)
    print(f'  Original tokens : {q_orig}')
    print(f'  Added terms     : {new_terms}')
    return _to_result_list(_score_docs(expanded_tokens), top_k)


# ── Demo ──────────────────────────────────────────────────────────────────────
display_results(
    search_bert('الصبر الصلاة', top_k=10),
    'BERT Expansion  |  "الصبر الصلاة"'
)

---
## Section 9 — Unified Search API & Method Comparison

Single `search()` entry point for Phase 3 (UI).

In [ ]:
def search(
    query  : str,
    method : str = 'or',
    top_k  : int = 10,
    **kwargs,
) -> List[Dict]:
    """
    Unified search API — entry point for Phase 3.

    Args:
        query  : Arabic query string
        method : 'or' | 'and' | 'rocchio' | 'bert'
        top_k  : number of results to return
        **kwargs: forwarded to the expansion function
                  rocchio — k_rel (int), n_terms (int), beta (float)
                  bert    — k_terms (int), threshold (float)

    Returns:
        List[Dict] with keys: doc_id, surah_id, surah_name, ayah_id, text, score
    """
    _dispatch = {
        'or'      : search_or,
        'and'     : search_and,
        'rocchio' : search_rocchio,
        'bert'    : search_bert,
    }
    fn = _dispatch.get(method)
    if fn is None:
        raise ValueError(f"Unknown method '{method}'. Choose from: {list(_dispatch)}.")
    return fn(query, top_k=top_k, **kwargs)


def compare_methods(query: str, top_k: int = 5) -> None:
    """Run all four methods on the same query and display results side-by-side."""
    print(f'\n{"="*110}')
    print(f'  QUERY: "{query}"')
    print(f'{"="*110}')
    for method in ('or', 'and', 'rocchio', 'bert'):
        results = search(query, method=method, top_k=top_k)
        display_results(results, method.upper())


# ── Demo ──────────────────────────────────────────────────────────────────────
compare_methods('النور والهداية', top_k=5)

In [ ]:
# Save pre-computed BERT vocabulary embeddings for Phase 3 (avoids re-encoding)
np.save(os.path.join(OUTPUT_DIR, 'vocab_vecs.npy'),  VOCAB_VECS)
with open(os.path.join(OUTPUT_DIR, 'vocab_terms.json'), 'w', encoding='utf-8') as f:
    json.dump(VOCAB_TERMS, f, ensure_ascii=False)

print(f'Saved vocab_vecs.npy   shape={VOCAB_VECS.shape}  dtype={VOCAB_VECS.dtype}')
print(f'Saved vocab_terms.json ({len(VOCAB_TERMS):,} terms)')
print(f'\nAll Phase 2 artifacts saved to ./{OUTPUT_DIR}/')

---
## Summary

### Phase 1 — Indexing
```
quran.json
  │  load_quran + validate_structure
  │  build_corpus          → 6 236 ayah documents
  │  preprocess()          → diacritics → normalize → tokenize → stopwords → ISRI stem
  └─ build_inverted_index  → term → { df, postings: {doc_id: tf} }
       └─ saved artifacts: c_quran.json · inverted_index.json · vocab.json · doc_metadata.json
```

### Phase 2 — Retrieval & Query Expansion
```
User query
  │  preprocess()  (identical pipeline → tokens match the index)
  │
  ├─ [or]      search_or()       TF-IDF, union of all postings
  ├─ [and]     search_and()      posting-list intersection + TF-IDF re-rank
  ├─ [rocchio] search_rocchio()  Rocchio PRF centroid expansion + TF-IDF
  └─ [bert]    search_bert()     multilingual SBERT cosine expansion + TF-IDF
          └─ unified:  search(query, method='or'|'and'|'rocchio'|'bert', top_k=10)
```

| Function | Input | Returns |
|----------|-------|---------|
| `search_or(query)` | Arabic text | top-k results (TF-IDF OR) |
| `search_and(query)` | Arabic text | top-k results (TF-IDF AND) |
| `search_rocchio(query)` | Arabic text | top-k results (Rocchio PRF) |
| `search_bert(query)` | Arabic text | top-k results (BERT expansion) |
| `search(query, method, top_k)` | Arabic text | unified dispatch |
| `compare_methods(query)` | Arabic text | side-by-side comparison |